In [2]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json

def age_to_group(age):

    if age < 18:
        return "0-18"
    elif age <= 40:
        return "18-40"
    elif age <= 60:
        return "40-60"
    elif age <= 80:
        return "60-80"
    else:
        return "80+"



source_dir = Path("/home/vito/ibrahimm/projects/AI4Health/sourcedata/Chest Xray/physionet.org/files/mimic-cxr-jpg/2.0.0/")
raw_dir = Path("/home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/my_work/raw_data")
resize_size = 512

original_metadata_df = pd.read_csv(raw_dir / "mimic-cxr-2.0.0-metadata-with-demographics.csv")
original_metadata_df['age_group'] = original_metadata_df['anchor_age'].apply(age_to_group)

chexpert_df = pd.read_csv(raw_dir / "mimic-cxr-2.0.0-chexpert.csv")
impressions_df = pd.read_csv("/home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/RoentGen-v2/real_data/sections/mimic_cxr_sectioned.csv")
impressions_df['study_id'] = impressions_df['study'].str.replace('s', '').astype(int)
merged_df = pd.merge(original_metadata_df, chexpert_df, on=["subject_id", "study_id"], how="inner")
# roentgen-v2 use impressions
all_df = pd.merge(impressions_df, merged_df, on="study_id", how="inner")
all_df['image'] = all_df.apply(
    lambda row: f"{raw_dir}/files/p{str(int(row['subject_id']))[:2]}/p{str(int(row['subject_id']))}/"
    f"s{int(row['study_id'])}/{row['dicom_id']}.jpg", 
    axis=1)
all_df.shape


(377024, 36)

In [3]:
# --- Filter and prepare PA_data as before ---
PA_data = all_df[all_df['ViewPosition'] == 'PA']
print('Data after excluding non-PA studies:', PA_data.shape)
PA_data = PA_data[PA_data['impression'].notna()]
print('Data after excluding studies with no impression:', PA_data.shape)
ethnicity_map = {
    'WHITE': 'White',
    'HISPANIC/LATINO': 'Hispanic', 
    'BLACK/AFRICAN AMERICAN': 'Black',
    'ASIAN': 'Asian'
}
PA_data['ethnicity'] = PA_data['ethnicity'].map(ethnicity_map)
ethnicity_to_drop = ['UNKNOWN', 'OTHER', 'UNABLE TO OBTAIN','AMERICAN INDIAN/ALASKA NATIVE']
PA_data = PA_data.dropna(subset=['ethnicity'])
PA_data['impression_length'] = PA_data['impression'].str.len()
PA_data = PA_data[~PA_data['ethnicity'].isin(ethnicity_to_drop)]
print('Data after excluding specified ethnicity values:', PA_data.shape)
PA_data['gender'] = PA_data['gender'].map({'M': 'male', 'F': 'female'})
PA_data['sentence'] = PA_data.apply(lambda row: f"{int(row['anchor_age'])} year old {row['ethnicity']} {row['gender']}. {row['impression']}", axis=1)




Data after excluding non-PA studies: (96143, 36)
Data after excluding studies with no impression: (87056, 36)
Data after excluding specified ethnicity values: (70433, 37)


In [4]:
import tarfile

demo_tar_path = "/home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/RoentGen-v2/demo_data/demo_webdataset/demo_data_0.tar"

with tarfile.open(demo_tar_path, "r") as tar:
    print("Contents of", demo_tar_path)
    for member in tar.getmembers():
        print(member.name)


Contents of /home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/RoentGen-v2/demo_data/demo_webdataset/demo_data_0.tar
demo0002.prompt_metadata
demo0002.pt_image
demo0003.prompt_metadata
demo0003.pt_image
demo0004.prompt_metadata
demo0004.pt_image
demo0005.prompt_metadata
demo0005.pt_image
demo0007.prompt_metadata
demo0007.pt_image
demo0008.prompt_metadata
demo0008.pt_image
demo0009.prompt_metadata
demo0009.pt_image


In [40]:
import tarfile

demo_tar_path = "/home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/my_work/raw_data/mimiccxr_pa_sentence_webdataset.tar"

with tarfile.open(demo_tar_path, "r") as tar:
    print("Contents of", demo_tar_path)
    for member in tar.getmembers():
        print(member.name)


Contents of /home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/my_work/raw_data/mimiccxr_pa_sentence_webdataset.tar
000000.pt_image
000000.prompt_metadata
000003.pt_image
000003.prompt_metadata
000022.pt_image
000022.prompt_metadata
000025.pt_image
000025.prompt_metadata
000027.pt_image
000027.prompt_metadata
000031.pt_image
000031.prompt_metadata
000038.pt_image
000038.prompt_metadata
000040.pt_image
000040.prompt_metadata
000049.pt_image
000049.prompt_metadata
000052.pt_image
000052.prompt_metadata
000053.pt_image
000053.prompt_metadata
000054.pt_image
000054.prompt_metadata
000057.pt_image
000057.prompt_metadata
000063.pt_image
000063.prompt_metadata
000065.pt_image
000065.prompt_metadata
000066.pt_image
000066.prompt_metadata
000074.pt_image
000074.prompt_metadata
000076.pt_image
000076.prompt_metadata
000077.pt_image
000077.prompt_metadata
000079.pt_image
000079.prompt_metadata
000081.pt_image
000081.prompt_metadata
000084.pt_image
000084

In [39]:
# --- Create WebDataset from PA_data image/sentence pairs in new format ---

import os
import torch
from PIL import Image
from tqdm import tqdm
import numpy as np
# Directory where original images are stored
img_dir = source_dir / "files"

# Output tar file for WebDataset (rename if desired)
output_tar = raw_dir / "mimiccxr_pa_sentence_webdataset.tar"
size_txt = raw_dir / "mimiccxr_pa_sentence_webdataset_size.txt"

import tarfile
import io

count = 0

with tarfile.open(output_tar, "w") as tar:
    for idx, row in tqdm(PA_data.iterrows(), total=len(PA_data), desc="Writing WebDataset"):
        img_path = Path(row['image'])
        if img_path.exists():
            try:
                # 1. Encode image as PyTorch tensor (uint8, pickled)
                with Image.open(img_path) as img:
                    img = img.convert("RGB")
                    tensor = torch.from_numpy(np.array(img))  # shape: (H, W, 3)
                    if tensor.dtype != torch.uint8:
                        tensor = tensor.to(torch.uint8)
                img_bytes = io.BytesIO()
                torch.save(tensor, img_bytes)
                img_bytes.seek(0)

                # 2. Prompt metadata (UTF-8)
                prompt_bytes = row['sentence'].encode("utf-8")
                prompt_stream = io.BytesIO(prompt_bytes)

                # 3. Generate keys (webdataset standard: 000001, 000002, ...)
                key = f"{idx:06d}"

                # 4. Add .pt_image file
                ptinfo = tarfile.TarInfo(f"{key}.pt_image")
                ptinfo.size = img_bytes.getbuffer().nbytes
                img_bytes.seek(0)
                tar.addfile(ptinfo, img_bytes)

                # 5. Add .prompt_metadata file
                promptinfo = tarfile.TarInfo(f"{key}.prompt_metadata")
                promptinfo.size = prompt_stream.getbuffer().nbytes
                prompt_stream.seek(0)
                tar.addfile(promptinfo, prompt_stream)

                count += 1
            except Exception as e:
                print(f"Could not process image {img_path}: {e}")
        else:
            print(f"Missing image file: {img_path}")

print(f"Total samples written: {count}")

# Write count to _size.txt file
with open(size_txt, "w") as f:
    f.write(str(count) + "\n")



Writing WebDataset: 100%|██████████| 70433/70433 [06:20<00:00, 185.12it/s]

Total samples written: 70433


# Tokenizer

In [ ]:
# Tokenize impressions in PA_data using Stable Diffusion's tokenizer
from transformers import AutoTokenizer

# Choose the tokenizer consistent with training code (CLIP tokenizer from SD v1-4)
model_id = "stanfordmimi/RoentGen-v2"
try:
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        subfolder="tokenizer",
        use_fast=True,
        trust_remote_code=True,
        token=token,
    )
except Exception:
    # Fallback to default loading if subfolder is not available
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True, token=token)

# Ensure PA_data exists and has an 'impression' column
assert 'PA_data' in globals(), "PA_data DataFrame not found in the notebook."
assert 'impression' in PA_data.columns, "PA_data is missing the 'impression' column."

# Prepare texts (handle NaNs)
texts = PA_data['impression'].fillna("").astype(str)

# Get tokenizer max length (defaults to 77 for CLIP)
if hasattr(tokenizer, 'model_max_length'):
    max_len = tokenizer.model_max_length
else:
    max_len = 77  # fallback CLIP default

# Batch tokenize for efficiency (with truncation and max_length)
encodings = tokenizer(
    texts.tolist(),
    padding='max_length',
    truncation=True,
    max_length=max_len,
    return_attention_mask=True,
)

# Store tokenized outputs as list columns
PA_data['input_ids'] = encodings['input_ids']
PA_data['attention_mask'] = encodings['attention_mask']

# Check for entries that exceeded the token limit BEFORE truncation (by re-tokenizing with no truncation)
lengths = [len(tokenizer.encode(t, add_special_tokens=True, truncation=False)) for t in texts]
PA_data['n_input_tokens'] = lengths
PA_data['truncated'] = PA_data['n_input_tokens'] > max_len

# Show stats about truncation
n_truncated = PA_data['truncated'].sum()
print(f"Number of samples truncated due to token limit ({max_len}): {n_truncated}")
if n_truncated > 0:
    print(PA_data.loc[PA_data['truncated'], ['impression', 'n_input_tokens']].head())

# Quick sanity check: show shapes/lengths of the first few entries
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Max length used:", max_len)
print("First input_ids length:", len(PA_data['input_ids'].iloc[0]) if len(PA_data) else None)


Token indices sequence length is longer than the specified maximum sequence length for this model (80 > 77). Running this sequence through the model will result in indexing errors


Number of samples truncated due to token limit (77): 3939
                                            impression  n_input_tokens
25   Compared to chest radiographs since ___, most ...              80
335  PA and lateral chest reviewed in the absence o...             123
574  Compared to chest radiographs ___ and chest CT...              91
709  Pacemaker leads terminate in right atrium and ...              89
744  PA and lateral chest compared to ___.\n \n Sli...              79
Tokenizer vocab size: 49408
Max length used: 77
First input_ids length: 77


# Tokenizer

In [88]:
PA_data[['gender', 'ethnicity', 'age_group']].value_counts()

gender  ethnicity               age_group
M       WHITE                   60-80        11148
                                40-60         9688
F       WHITE                   60-80         8084
                                40-60         6804
        BLACK/AFRICAN AMERICAN  40-60         3908
        WHITE                   18-40         3143
M       WHITE                   18-40         2938
        BLACK/AFRICAN AMERICAN  40-60         2819
F       BLACK/AFRICAN AMERICAN  60-80         2733
M       WHITE                   80+           2594
F       WHITE                   80+           2487
        BLACK/AFRICAN AMERICAN  18-40         2187
M       BLACK/AFRICAN AMERICAN  60-80         1872
F       HISPANIC/LATINO         40-60         1399
M       HISPANIC/LATINO         40-60         1182
        BLACK/AFRICAN AMERICAN  18-40         1051
F       HISPANIC/LATINO         18-40          875
                                60-80          828
M       HISPANIC/LATINO         60-80   

In [72]:
PA_data['ethnicity'].isna().sum()


np.int64(0)

In [76]:
PA_data[PA_data['impression_length']<12]


,study,impression,findings,last_paragraph,comparison,study_id,dicom_id,subject_id,PerformedProcedureStepDescription,ViewPosition,...,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices,impression_length
117875,s55216169,Stable exam,"Biapical pleural thickening, stable. Subtle l...",NaN,Chest radiographs ___,55216169,02093259-5924b437-068eadf8-c3e6f2d5-29bdd2f0,13141248,CHEST (PA AND LAT),PA,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,11
117878,s55216169,Stable exam,"Biapical pleural thickening, stable. Subtle l...",NaN,Chest radiographs ___,55216169,4fcaa500-471b8aa9-303ffc38-60dd5c0e-3e45f0dd,13141248,CHEST (PA AND LAT),PA,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,11
157924,s52980564,Mild edema.,PA and lateral views of the chest provided. ...,NaN,___,52980564,014eac91-7429718c-19a19d69-c3b1b813-82efff02,14202902,CHEST (PA AND LAT),PA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
157925,s52980564,Mild edema.,PA and lateral views of the chest provided. ...,NaN,___,52980564,0861eb49-3e51b13f-74f50b11-b328dd60-870285cf,14202902,CHEST (PA AND LAT),PA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
170961,s53267359,No failure.,The heart and mediastinum are normal. The lun...,NaN,NaN,53267359,5449e5e9-ad8da8bc-32876dfa-77e526da-61070339,14557977,Performed Desc,PA,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,11
203960,s57139108,Clear lungs,"No focal consolidation, pleural effusion or pn...",NaN,None available,57139108,12a0b811-0c3aecdb-288610f4-0d17c181-5cac6074,15407174,Performed Desc,PA,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,11
246212,s58508752,Stable exam,"Extensive bilateral pulmonary infiltrates, con...",NaN,___,58508752,0b78d19b-0ea217d7-8545e48f-4677288a-b968492a,16521833,Performed Desc,PA,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,11
326923,s53913472,Mild edema.,PA and lateral views of the chest provided. ...,NaN,NaN,53913472,96eaa39d-4bc0be4a-e67e7d41-30d19f3a-ad1c3ea0,18673777,CHEST (PA AND LAT),PA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
337594,s52901581,Normal CXR.,"Heart, mediastinum and the lung fields appear ...",NaN,NaN,52901581,6adebd41-cde5cad8-587f031c-1cc06861-8e372901,18954833,Performed Desc,PA,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,11


In [ ]:


# Drop rows where view position is not PA or AP
view_position_to_keep = ['AP','PA','LATERAL', 'LL']
metadata_df = metadata_df.dropna(subset=['ViewPosition'])
metadata_df = metadata_df[metadata_df['ViewPosition'].isin(view_position_to_keep)]






# # Replace ethnicity values with simplified categories
# ethnicity_mapping = {
#     'WHITE': 'white',
#     'HISPANIC/LATINO': 'Hispanic', 
#     'BLACK/AFRICAN AMERICAN': 'Black',
#     'AMERICAN INDIAN/ALASKA NATIVE': 'Native'
# }

# merged_df['ethnicity'] = merged_df['ethnicity'].map(ethnicity_mapping)


# Create stratification key
merged_df["stratify_key"] = merged_df.apply(stratify_group, axis=1)


In [1]:

import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm

def age_to_group(age):
    if pd.isna(age):
        return "unknown age group"
    age = float(age)
    if age < 20:
        return "0-20"
    elif age <= 40:
        return "20-40"
    elif age <= 60:
        return "40-60"
    elif age <= 80:
        return "60-80"
    else:
        return "80+"

def generate_sentence(row, chexpert_labels):
    view = row.get("ViewPosition", "unknown view")
    age_group = age_to_group(row.get("anchor_age", None)) if pd.notna(row.get("anchor_age")) else "unknown age group"
    sex = str(row.get("gender", "unknown gender")).lower() if pd.notna(row.get("gender")) else "unknown gender"
    sex_raw = str(row.get("gender", "")).strip().upper()
    if sex_raw == "F":
        sex = "female"
    elif sex_raw == "M":
        sex = "male"
    else:
        sex = "unknown gender"

    ethnicity = str(row.get("ethnicity", "unknown ethnicity")).lower() if pd.notna(row.get("ethnicity")) else "unknown ethnicity"

    base = f"This is a {view} view X-ray of {sex} patient aged between {age_group} of {ethnicity} ethnicity."
    findings = []

    for label in chexpert_labels:
        if row.get(label) == 1.0:
            label_text = label.lower()
            if label_text in ["atelectasis", "edema", "enlarged cardiomediastinum"]:
                findings.append(f"an {label_text}")
            else:
                findings.append(f"a {label_text}")

    if findings:
        findings_sentence = " The patient has " + ", ".join(findings) + "."
    else:
        findings_sentence = ""

    return base + findings_sentence


In [3]:
chexpert_df

,subject_id,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices
0,10000032,50414267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1,10000032,53189527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
2,10000032,53911762,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
3,10000032,56699142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
4,10000764,57375967,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227822,19999442,58708861,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0
227823,19999733,57132437,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
227824,19999987,55368167,1.0,-1.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN
227825,19999987,58621812,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


# raw

In [5]:
# load in mapping file
mimic_cxr_jpg_path = Path("/mnthpc/netapp01/projects/AI4Health/sourcedata/Chest Xray/physionet.org/files/mimic-cxr-jpg/2.0.0/")
mimic_cxr_path = Path("/mnthpc/netapp01/projects/AI4Health/sourcedata/Chest Xray/physionet.org/files/mimic-cxr/2.0.0/")

df = pd.read_csv(mimic_cxr_path / 'cxr-record-list.csv.gz', header=0, sep=',')

n = df.shape[0]
print(f'{n} DICOMs in MIMIC-CXR v2.0.0.')

n = df['study_id'].nunique()
print(f'  {n} studies.')

n = df['subject_id'].nunique()
print(f'  {n} subjects.')

dicoms = set(df['dicom_id'].tolist())



377110 DICOMs in MIMIC-CXR v2.0.0.
  227835 studies.
  65379 subjects.


In [6]:

df_split = pd.read_csv(mimic_cxr_jpg_path / 'mimic-cxr-2.0.0-split.csv.gz')
df_metadata = pd.read_csv(mimic_cxr_jpg_path / 'mimic-cxr-2.0.0-metadata.csv.gz')
chexpert_df = pd.read_csv(raw_dir / "mimic-cxr-2.0.0-chexpert.csv")



In [8]:
merged_df = pd.merge(df_metadata, chexpert_df, on=["subject_id", "study_id"], how="inner")
merged_df['ViewPosition'].value_counts()

ViewPosition
AP                147169
PA                 96155
LATERAL            82852
LL                 35129
PA LLD                 4
LAO                    3
RAO                    3
AP AXIAL               2
AP LLD                 2
XTABLE LATERAL         2
AP RLD                 2
SWIMMERS               1
PA RLD                 1
LPO                    1
Name: count, dtype: int64

In [38]:
df_metadata['ViewPosition'].value_counts()

ViewPosition
AP                147173
PA                 96161
LATERAL            82853
LL                 35133
PA LLD                 4
LAO                    3
RAO                    3
AP AXIAL               2
AP LLD                 2
XTABLE LATERAL         2
AP RLD                 2
SWIMMERS               1
PA RLD                 1
LPO                    1
Name: count, dtype: int64